# X-Ray Baggage Screening — Analysis Notebook
**ResNet-50 · SIXray Dataset · Grad-CAM**

Run cells in order after completing training (`python src/train.py --demo`)

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from pathlib import Path

# Ensure imports from src/
sys.path.insert(0, os.path.abspath('..'))
from src.utils import load_config
cfg = load_config('../config.yaml')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Config loaded:', cfg['model']['backbone'])

## 1. Dataset Exploration (EDA)

In [ ]:
# Count images per class
raw_root = Path('../') / cfg['dataset']['root']
pos_dir = raw_root / 'positive'
neg_dir = raw_root / 'negative'

exts = {'.jpg','.jpeg','.png','.bmp'}
pos_files = [f for f in pos_dir.glob('*') if f.suffix.lower() in exts]
neg_files = [f for f in neg_dir.glob('*') if f.suffix.lower() in exts]

print(f'Positive (prohibited): {len(pos_files):,}')
print(f'Negative (safe):       {len(neg_files):,}')
print(f'Imbalance ratio:       1 : {len(neg_files)/max(len(pos_files),1):.1f}')

# Bar chart
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Prohibited', 'Safe'], [len(pos_files), len(neg_files)],
       color=['#F44336', '#4CAF50'], width=0.5)
ax.set_title('Class Distribution', fontweight='bold')
ax.set_ylabel('Image Count')
for bar, n in zip(ax.patches, [len(pos_files), len(neg_files)]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{n:,}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Display sample images from each class
import random
n_show = 8
pos_sample = random.sample(pos_files, min(n_show, len(pos_files)))
neg_sample = random.sample(neg_files, min(n_show, len(neg_files)))

fig, axes = plt.subplots(2, n_show, figsize=(n_show*2, 5))
fig.suptitle('Sample X-Ray Images', fontsize=13, fontweight='bold')
for i, (pos, neg) in enumerate(zip(pos_sample, neg_sample)):
    axes[0, i].imshow(Image.open(pos).resize((112,112)), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(Image.open(neg).resize((112,112)), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Prohibited', fontsize=11, color='#F44336')
axes[1, 0].set_ylabel('Safe', fontsize=11, color='#4CAF50')
plt.tight_layout(); plt.show()

In [ ]:
# Pixel intensity histograms
def sample_pixels(files, n=5):
    vals = []
    for f in random.sample(files, min(n, len(files))):
        arr = np.array(Image.open(f).convert('L'))
        vals.extend(arr.flatten().tolist())
    return vals

pos_px = sample_pixels(pos_files, 10)
neg_px = sample_pixels(neg_files, 10)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pos_px, bins=60, alpha=0.6, color='#F44336', label='Prohibited', density=True)
ax.hist(neg_px, bins=60, alpha=0.6, color='#4CAF50', label='Safe', density=True)
ax.set_xlabel('Pixel Intensity'); ax.set_ylabel('Density')
ax.set_title('Pixel Intensity Distribution')
ax.legend(); plt.tight_layout(); plt.show()

## 2. Training Curves

In [ ]:
log_csv = Path('../') / cfg['outputs']['log_dir'] / 'training_log.csv'
if not log_csv.exists():
    print('No training log found. Run: python src/train.py --demo')
else:
    df = pd.read_csv(log_csv)
    print(df.tail(5).to_string(index=False))

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle('Training Curves — ResNet-50 X-Ray Classifier',
                 fontsize=13, fontweight='bold')

    def plot_metric(ax, col_train, col_val, title, ylabel):
        ax.plot(df['epoch'], df[col_train], label='Train', color='#2196F3', lw=2)
        ax.plot(df['epoch'], df[col_val],   label='Val',   color='#FF5722', lw=2)
        # shade phase boundary
        fe = cfg['model']['freeze_epochs']
        ax.axvline(fe, ls='--', color='grey', lw=1, label=f'Unfreeze (ep {fe})')
        ax.set_title(title); ax.set_ylabel(ylabel); ax.set_xlabel('Epoch')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plot_metric(axes[0,0], 'train_loss', 'val_loss',  'Loss',      'Loss')
    plot_metric(axes[0,1], 'train_acc',  'val_acc',   'Accuracy',  'Accuracy')
    plot_metric(axes[1,0], 'train_f1',   'val_f1',    'F1 Score',  'F1')
    plot_metric(axes[1,1], 'train_rec',  'val_rec',   'Recall (Prohibited)', 'Recall')

    plt.tight_layout()
    plt.savefig('../outputs/visualizations/training_curves.png', dpi=150)
    plt.show()
    print('Saved: outputs/visualizations/training_curves.png')

## 3. Evaluation Metrics

In [ ]:
report_path = Path('../') / cfg['outputs']['viz_dir'] / 'metrics_report.json'
if not report_path.exists():
    print('No metrics report. Run: python src/evaluate.py')
else:
    with open(report_path) as f:
        metrics = json.load(f)

    print('=== Summary Metrics ===')
    print(f"ROC-AUC:            {metrics['roc_auc']:.4f}")
    print(f"Average Precision:  {metrics['average_precision']:.4f}")
    print(f"Best Threshold:     {metrics['best_threshold']:.2f}")
    print(f"Best F1 @ thresh:   {metrics['best_f1_at_thresh']:.4f}")
    print()

    rows = []
    for cls in ['Safe', 'Prohibited']:
        d = metrics['per_class'].get(cls, {})
        rows.append({'Class': cls,
                     'Precision': d.get('precision',0),
                     'Recall':    d.get('recall',0),
                     'F1':        d.get('f1-score',0),
                     'Support':   int(d.get('support',0))})
    print(pd.DataFrame(rows).to_string(index=False))

## 4. Grad-CAM Visualizations

In [ ]:
gradcam_path = Path('../') / cfg['outputs']['viz_dir'] / 'gradcam_grid.png'
if gradcam_path.exists():
    img = Image.open(gradcam_path)
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Grad-CAM — Model Attention on Test Set', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('No Grad-CAM grid. Run: python src/gradcam.py')

## 5. Confusion Matrix + ROC

In [ ]:
viz_dir = Path('../') / cfg['outputs']['viz_dir']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plots = ['confusion_matrix.png', 'roc_curve.png', 'pr_curve.png']
titles = ['Confusion Matrix', 'ROC Curve', 'PR Curve']
for ax, pname, title in zip(axes, plots, titles):
    p = viz_dir / pname
    if p.exists():
        ax.imshow(Image.open(p))
    else:
        ax.text(0.5, 0.5, f'Missing\n{pname}', ha='center', va='center',
                transform=ax.transAxes)
    ax.set_title(title, fontweight='bold'); ax.axis('off')
plt.tight_layout(); plt.show()